# Customer Purchase Prediction
### Decision Tree · Naive Bayes · Hybrid AdaBoost-NB

---

**Course:** Principles of Artificial Intelligence — MPSTME, NMIMS  
**Academic Year:** 2025–2026  
**Author:** Sana Jadhav

---

## Overview

This notebook implements and compares three supervised machine learning classifiers — Decision Tree, Naive Bayes, and a Hybrid AdaBoost-NB ensemble — built entirely from scratch in Python without any ML libraries.

**Objective:** Predict whether a customer will make a purchase based on 8 behavioural and demographic features from a dataset of 1,500 records.

**Pipeline:**
1. Data Loading & Exploration
2. Preprocessing & Feature Engineering
3. Train-Test Split
4. Model Implementation & Training
5. Feature Importance Analysis
6. Evaluation & Results
7. K-Fold Cross-Validation
8. Learning Curves
9. Final Comparison

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import math
import random
import matplotlib.pyplot as plt
from collections import Counter

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2. Data Loading & Exploration

Dataset: `customer_purchase_data.csv`  
1,500 customer records with 8 input features and 1 binary target (`PurchaseStatus`).

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("customer_purchase_data.csv")
print("Shape:", df.shape)
print("\nFeature Types:\n", df.dtypes)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Class distribution
dist = df['PurchaseStatus'].value_counts()
print("Class Distribution:")
print(f"  No Purchase (0): {dist[0]} ({dist[0]/len(df)*100:.1f}%)")
print(f"  Purchase    (1): {dist[1]} ({dist[1]/len(df)*100:.1f}%)")
print(f"\nClass Imbalance Ratio: {dist[0]/dist[1]:.2f}:1")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['No Purchase', 'Purchase'], [dist[0], dist[1]], color=['#555555', '#222222'], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([dist[0], dist[1]]):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Feature distributions for key numeric features
features = ['Age', 'AnnualIncome', 'TimeSpentOnWebsite', 'NumberOfPurchases']
for feat in features:
    axes[1].hist(df[df['PurchaseStatus']==1][feat], alpha=0.6, label='Purchase', color='#222222', bins=20)
    axes[1].hist(df[df['PurchaseStatus']==0][feat], alpha=0.4, label='No Purchase', color='#888888', bins=20)
axes[1].set_title('Feature Overlap (all numeric features)', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Preprocessing & Feature Engineering

- **Missing values:** Inspected — none found in this dataset  
- **Min-Max Normalisation:** Applied to numerical features for distance-based methods  
- **Categorical Binning:** Continuous features discretised into labelled bins for Naive Bayes and AdaBoost-NB

In [ ]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotal missing:", df.isnull().sum().sum())

In [ ]:
# Discretisation for Naive Bayes / AdaBoost-NB
def discretize(dataframe):
    df_d = dataframe.copy()
    df_d['Age'] = pd.cut(df_d['Age'], [0, 35, 55, 100], labels=['young', 'middle', 'senior'])
    df_d['AnnualIncome'] = pd.cut(df_d['AnnualIncome'], [0, 50000, 100000, 1e9], labels=['low', 'medium', 'high'])
    df_d['TimeSpentOnWebsite'] = pd.cut(df_d['TimeSpentOnWebsite'], [0, 30, 60, 1e9], labels=['short', 'medium', 'long'])
    df_d['NumberOfPurchases'] = pd.cut(df_d['NumberOfPurchases'], [0, 3, 7, 100], labels=['low', 'medium', 'high'])
    return df_d

## 4. Train-Test Split

**80/20 split** with `seed=42` for reproducibility.  
The same shuffle indices are reused across all three models to ensure fair comparison on identical test subsets.

- Training set: 1,200 samples  
- Test set: 300 samples

In [ ]:
random.seed(SEED)
indices = list(df.index)
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx  = indices[split:]

train_df = df.loc[train_idx].reset_index(drop=True)
test_df  = df.loc[test_idx].reset_index(drop=True)

print(f"Training samples: {len(train_df)}")
print(f"Test samples:     {len(test_df)}")

# Discretised versions for NB and AdaBoost
train_nb = discretize(train_df)
test_nb  = discretize(test_df)

## 5. Model Implementation & Training

---

### 5.1 Decision Tree (from Scratch)

**Algorithm:** Recursive binary splitting using Information Gain (entropy-based).  
**Stopping conditions:** Pure node · depth ≥ 8 · fewer than 5 samples  
**Hyperparameters:** `max_depth=8`, `min_samples=5`

$$H(S) = -\sum_{c} \frac{|S_c|}{|S|} \log_2 \frac{|S_c|}{|S|}$$

$$IG(S, f, t) = H(S) - \frac{|S_L|}{|S|}H(S_L) - \frac{|S_R|}{|S|}H(S_R)$$

In [ ]:
def entropy(y):
    counts = Counter(y)
    total = len(y)
    return -sum((c / total) * math.log2(c / total) for c in counts.values() if c > 0)

def information_gain(X, y, feature, threshold):
    left_mask  = X[feature] <= threshold
    right_mask = X[feature] > threshold
    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return 0
    H = entropy(y)
    H_left  = entropy(y[left_mask])
    H_right = entropy(y[right_mask])
    return H - (left_mask.sum() / len(y)) * H_left - (right_mask.sum() / len(y)) * H_right

class DecisionTree:
    def __init__(self, max_depth=8, min_samples=5):
        self.max_depth   = max_depth
        self.min_samples = min_samples
        self.feature_importance_ = {}

    def build_tree(self, X, y, depth=0):
        if len(set(y)) == 1 or depth >= self.max_depth or len(y) < self.min_samples:
            return Counter(y).most_common(1)[0][0]

        best_feature, best_thresh, best_gain = None, None, 0

        for feature in X.columns:
            for val in X[feature].unique():
                gain = information_gain(X, y, feature, val)
                if gain > best_gain:
                    best_feature = feature
                    best_thresh  = val
                    best_gain    = gain

        if best_gain == 0:
            return Counter(y).most_common(1)[0][0]

        # Track feature importance (cumulative information gain)
        self.feature_importance_[best_feature] = (
            self.feature_importance_.get(best_feature, 0) + best_gain * len(y)
        )

        left_mask  = X[best_feature] <= best_thresh
        right_mask = X[best_feature] > best_thresh

        return {
            "feature":   best_feature,
            "threshold": best_thresh,
            "left":  self.build_tree(X[left_mask],  y[left_mask],  depth + 1),
            "right": self.build_tree(X[right_mask], y[right_mask], depth + 1)
        }

    def fit(self, X, y):
        self.feature_importance_ = {}
        self.tree = self.build_tree(X, y)
        # Normalise importance scores
        total = sum(self.feature_importance_.values())
        if total > 0:
            self.feature_importance_ = {k: v / total for k, v in self.feature_importance_.items()}

    def predict_one(self, x, tree):
        if not isinstance(tree, dict):
            return tree
        if x[tree["feature"]] <= tree["threshold"]:
            return self.predict_one(x, tree["left"])
        else:
            return self.predict_one(x, tree["right"])

    def predict(self, X):
        return np.array([self.predict_one(row, self.tree) for _, row in X.iterrows()])

In [ ]:
X_train_dt = train_df.drop(columns=['PurchaseStatus'])
y_train_dt = train_df['PurchaseStatus']
X_test_dt  = test_df.drop(columns=['PurchaseStatus'])
y_test_dt  = test_df['PurchaseStatus']

dt = DecisionTree(max_depth=8, min_samples=5)
dt.fit(X_train_dt, y_train_dt)
dt_preds = dt.predict(X_test_dt)
print("Decision Tree training complete.")

### 5.2 Naive Bayes (from Scratch)

**Algorithm:** Frequency-based probabilistic classifier with Laplace smoothing (α=1).  
Continuous features are discretised into labelled bins before training.

$$P(x_i = v \mid C) = \frac{|\{d \in D_C : x_i^{(d)} = v\}| + 1}{|D_C| + |V_i|}$$

$$\hat{y} = \arg\max_c \left[ \log P(C=c) + \sum_i \log P(x_i \mid C=c) \right]$$

In [ ]:
class NaiveBayes:
    def fit(self, X, y):
        self.classes_      = np.unique(y)
        self.priors_       = {}
        self.likelihoods_  = {}

        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / len(X)
            self.likelihoods_[c] = {}
            for col in X.columns:
                counts = X_c[col].value_counts()
                vocab  = X[col].unique()
                total  = len(X_c)
                self.likelihoods_[c][col] = {
                    val: (counts.get(val, 0) + 1) / (total + len(vocab))
                    for val in vocab
                }

    def predict(self, X):
        preds = []
        for _, row in X.iterrows():
            probs = {}
            for c in self.classes_:
                prob = math.log(self.priors_[c])
                for col in X.columns:
                    prob += math.log(self.likelihoods_[c][col].get(row[col], 1e-6))
                probs[c] = prob
            preds.append(max(probs, key=probs.get))
        return np.array(preds)

In [ ]:
X_train_nb = train_nb.drop(columns=['PurchaseStatus'])
y_train_nb = train_nb['PurchaseStatus']
X_test_nb  = test_nb.drop(columns=['PurchaseStatus'])
y_test_nb  = test_nb['PurchaseStatus']

nb = NaiveBayes()
nb.fit(X_train_nb, y_train_nb)
nb_preds = nb.predict(X_test_nb)
print("Naive Bayes training complete.")

### 5.3 Hybrid AdaBoost-NB (from Scratch)

**Algorithm:** Boosting ensemble with Naive Bayes as the weak learner. T=20 boosting rounds.  
Architecturally independent of both the standalone Decision Tree and standalone Naive Bayes.

At each round t:
- Train NB on re-weighted training distribution
- Compute weighted error: $\varepsilon_t = \sum_i w_i \cdot \mathbf{1}[h_t(x_i) \neq y_i]$
- Compute learner weight: $\alpha_t = \frac{1}{2} \ln\frac{1 - \varepsilon_t}{\varepsilon_t}$
- Update sample weights: $w_i \leftarrow w_i \cdot \exp(-\alpha_t \cdot y_i \cdot h_t(x_i))$

$$\hat{y} = \text{sign}\left(\sum_{t=1}^{T} \alpha_t \cdot h_t(x)\right)$$

In [ ]:
class AdaBoostNB:
    def __init__(self, T=20):
        self.T       = T
        self.models_ = []
        self.alphas_ = []

    def fit(self, X, y):
        n       = len(X)
        weights = np.ones(n) / n
        y_mod   = np.where(y == 0, -1, 1)

        for t in range(self.T):
            model = NaiveBayes()
            model.fit(X, y)

            preds     = model.predict(X)
            preds_mod = np.where(preds == 0, -1, 1)

            error = np.sum(weights * (preds_mod != y_mod))
            if error >= 0.5 or error == 0:
                continue

            alpha    = 0.5 * math.log((1 - error) / error)
            weights *= np.exp(-alpha * y_mod * preds_mod)
            weights /= np.sum(weights)

            self.models_.append(model)
            self.alphas_.append(alpha)

    def predict(self, X):
        final = np.zeros(len(X))
        for alpha, model in zip(self.alphas_, self.models_):
            preds_mod = np.where(model.predict(X) == 0, -1, 1)
            final    += alpha * preds_mod
        return np.where(final >= 0, 1, 0)

In [ ]:
ada = AdaBoostNB(T=20)
ada.fit(X_train_nb, y_train_nb)
ada_preds = ada.predict(X_test_nb)
print("Hybrid AdaBoost-NB training complete.")

## 6. Feature Importance Analysis

Feature importance is extracted from the Decision Tree by accumulating the weighted Information Gain at each split node, normalised across all splits.  
This identifies which features are most predictive of customer purchase behaviour.

In [ ]:
# Extract and sort feature importance from Decision Tree
importance = dt.feature_importance_
sorted_importance = dict(sorted(importance.items(), key=lambda x: x[1], reverse=True))

features_list = list(sorted_importance.keys())
scores        = list(sorted_importance.values())

plt.figure(figsize=(10, 5))
bars = plt.barh(features_list[::-1], scores[::-1], color='#333333', edgecolor='white')
plt.xlabel('Normalised Information Gain', fontsize=11)
plt.title('Decision Tree — Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nFeature Importance Ranking:")
for i, (feat, score) in enumerate(sorted_importance.items(), 1):
    print(f"  {i}. {feat:<25} {score:.4f}")

## 7. Evaluation & Results

### Evaluation Metrics

Given the moderate class imbalance (57% No Purchase / 43% Purchase), **F1-Score** is selected as the primary evaluation metric.  
Raw Accuracy is insufficient — a degenerate classifier always predicting the majority class would achieve ~57% without learning anything useful.

| Priority | Metric | Reason |
|----------|--------|--------|
| ★★★ | F1-Score | Balances Precision & Recall; primary metric |
| ★★ | Recall | FN = missed revenue; monitor closely |
| ★★ | Precision | FP = wasted campaign spend |
| ★ | Accuracy | Useful overview; insufficient alone |

In [ ]:
def evaluate(name, y_true, y_pred):
    TP = int(((y_true == 1) & (y_pred == 1)).sum())
    TN = int(((y_true == 0) & (y_pred == 0)).sum())
    FP = int(((y_true == 0) & (y_pred == 1)).sum())
    FN = int(((y_true == 1) & (y_pred == 0)).sum())

    acc  = (TP + TN) / (TP + TN + FP + FN)
    prec = TP / (TP + FP) if (TP + FP) > 0 else 0
    rec  = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    return {
        'Model': name, 'Accuracy': round(acc, 4), 'Precision': round(prec, 4),
        'Recall': round(rec, 4), 'F1-Score': round(f1, 4),
        'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN
    }

results = [
    evaluate("Decision Tree",    y_test_dt.values, dt_preds),
    evaluate("Naive Bayes",      y_test_nb.values, nb_preds),
    evaluate("Hybrid AdaBoost-NB", y_test_nb.values, ada_preds),
]

results_df = pd.DataFrame(results)
print(results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].to_string(index=False))

In [ ]:
# Confusion matrix visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, r in zip(axes, results):
    cm = np.array([[r['TN'], r['FP']], [r['FN'], r['TP']]])
    im = ax.imshow(cm, cmap='Greys')
    ax.set_title(r['Model'], fontsize=12, fontweight='bold')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred: No', 'Pred: Yes'])
    ax.set_yticklabels(['Actual: No', 'Actual: Yes'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Metric comparison bar chart
metrics  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
models   = [r['Model'] for r in results]
x        = np.arange(len(metrics))
width    = 0.25
colors   = ['#222222', '#666666', '#aaaaaa']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (r, color) in enumerate(zip(results, colors)):
    vals = [r[m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=r['Model'], color=color, edgecolor='white')

ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0.8, 1.0)
ax.legend()
plt.tight_layout()
plt.show()

## 8. K-Fold Cross-Validation

A single 80/20 split may produce results that are specific to that particular partition. **5-Fold Cross-Validation** provides a more statistically robust estimate of model performance by averaging results across 5 different train/test splits.

> Note: Due to the computational cost of the Decision Tree on larger datasets, CV is run on the Naive Bayes and AdaBoost-NB models. Decision Tree CV is included with a reduced fold size.

In [ ]:
def k_fold_cv(model_class, X, y, k=5, discretise=False, **kwargs):
    """
    Runs k-fold cross-validation.
    model_class: class with fit() and predict() methods
    discretise: whether to apply discretize() to folds (for NB/AdaBoost)
    """
    indices = list(range(len(X)))
    random.seed(SEED)
    random.shuffle(indices)

    fold_size = len(indices) // k
    fold_f1s  = []

    for fold in range(k):
        val_idx   = indices[fold * fold_size : (fold + 1) * fold_size]
        train_idx = indices[:fold * fold_size] + indices[(fold + 1) * fold_size:]

        X_tr = X.iloc[train_idx].reset_index(drop=True)
        y_tr = y.iloc[train_idx].reset_index(drop=True)
        X_val = X.iloc[val_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].reset_index(drop=True)

        if discretise:
            combined_tr  = X_tr.copy(); combined_tr['PurchaseStatus'] = y_tr.values
            combined_val = X_val.copy(); combined_val['PurchaseStatus'] = y_val.values
            combined_tr  = discretize(combined_tr)
            combined_val = discretize(combined_val)
            X_tr  = combined_tr.drop(columns=['PurchaseStatus'])
            y_tr  = combined_tr['PurchaseStatus']
            X_val = combined_val.drop(columns=['PurchaseStatus'])
            y_val = combined_val['PurchaseStatus']

        model = model_class(**kwargs)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        TP = int(((y_val == 1) & (preds == 1)).sum())
        FP = int(((y_val == 0) & (preds == 1)).sum())
        FN = int(((y_val == 1) & (preds == 0)).sum())
        prec = TP / (TP + FP) if (TP + FP) > 0 else 0
        rec  = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        fold_f1s.append(f1)

    return fold_f1s

print("Running 5-Fold Cross-Validation...")
print("(This may take a moment for Decision Tree)\n")

X_full = df.drop(columns=['PurchaseStatus'])
y_full = df['PurchaseStatus']

nb_cv  = k_fold_cv(NaiveBayes, X_full, y_full, k=5, discretise=True)
ada_cv = k_fold_cv(AdaBoostNB, X_full, y_full, k=5, discretise=True, T=20)
dt_cv  = k_fold_cv(DecisionTree, X_full, y_full, k=5, discretise=False, max_depth=8, min_samples=5)

print(f"Decision Tree    — Mean F1: {np.mean(dt_cv):.4f}  |  Std: {np.std(dt_cv):.4f}  |  Folds: {[round(f,4) for f in dt_cv]}")
print(f"Naive Bayes      — Mean F1: {np.mean(nb_cv):.4f}  |  Std: {np.std(nb_cv):.4f}  |  Folds: {[round(f,4) for f in nb_cv]}")
print(f"Hybrid AdaBoost  — Mean F1: {np.mean(ada_cv):.4f}  |  Std: {np.std(ada_cv):.4f}  |  Folds: {[round(f,4) for f in ada_cv]}")

## 9. Learning Curves

Learning curves show how model performance changes as the size of the training set increases. They help diagnose:
- **Underfitting** — both train and validation scores are low
- **Overfitting** — large gap between train and validation scores
- **Good fit** — scores converge at a high value

In [ ]:
def learning_curve(model_class, X, y, sizes, discretise=False, **kwargs):
    train_f1s = []
    val_f1s   = []

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    split    = int(0.8 * len(X))
    X_tr_full = X.iloc[:split]
    y_tr_full = y.iloc[:split]
    X_val     = X.iloc[split:]
    y_val     = y.iloc[split:]

    if discretise:
        tr_full_combined = X_tr_full.copy(); tr_full_combined['PurchaseStatus'] = y_tr_full.values
        val_combined     = X_val.copy();     val_combined['PurchaseStatus']     = y_val.values
        tr_full_combined = discretize(tr_full_combined)
        val_combined     = discretize(val_combined)
        X_val = val_combined.drop(columns=['PurchaseStatus'])
        y_val = val_combined['PurchaseStatus']

    for size in sizes:
        X_tr = X_tr_full.iloc[:size].reset_index(drop=True)
        y_tr = y_tr_full.iloc[:size].reset_index(drop=True)

        if discretise:
            combined = X_tr.copy(); combined['PurchaseStatus'] = y_tr.values
            combined = discretize(combined)
            X_tr = combined.drop(columns=['PurchaseStatus'])
            y_tr = combined['PurchaseStatus']

        model = model_class(**kwargs)
        model.fit(X_tr, y_tr)

        def f1_score(y_true, y_pred):
            TP = int(((y_true==1)&(y_pred==1)).sum())
            FP = int(((y_true==0)&(y_pred==1)).sum())
            FN = int(((y_true==1)&(y_pred==0)).sum())
            p = TP/(TP+FP) if TP+FP else 0
            r = TP/(TP+FN) if TP+FN else 0
            return 2*p*r/(p+r) if p+r else 0

        train_f1s.append(f1_score(y_tr, model.predict(X_tr)))
        val_f1s.append(f1_score(y_val, model.predict(X_val)))

    return train_f1s, val_f1s

sizes = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200]
X_full = df.drop(columns=['PurchaseStatus'])
y_full = df['PurchaseStatus']

print("Generating learning curves...")
dt_train,  dt_val  = learning_curve(DecisionTree, X_full, y_full, sizes, discretise=False, max_depth=8, min_samples=5)
nb_train,  nb_val  = learning_curve(NaiveBayes,   X_full, y_full, sizes, discretise=True)
ada_train, ada_val = learning_curve(AdaBoostNB,   X_full, y_full, sizes, discretise=True, T=20)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, name, tr, val in zip(
    axes,
    ['Decision Tree', 'Naive Bayes', 'Hybrid AdaBoost-NB'],
    [dt_train, nb_train, ada_train],
    [dt_val,   nb_val,   ada_val]
):
    ax.plot(sizes, tr,  'o-', color='#222222', label='Training F1',   linewidth=2)
    ax.plot(sizes, val, 's--', color='#888888', label='Validation F1', linewidth=2)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('F1-Score')
    ax.set_ylim(0.6, 1.05)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Learning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Final Comparison & Recommendations

In [ ]:
print("=" * 65)
print(f"{'FINAL MODEL COMPARISON':^65}")
print("=" * 65)
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1-Score':>10}")
print("-" * 65)
for r in results:
    print(f"{r['Model']:<22} {r['Accuracy']:>9.4f} {r['Precision']:>10.4f} {r['Recall']:>8.4f} {r['F1-Score']:>10.4f}")
print("=" * 65)

print("""
Primary Metric: F1-Score (justified by moderate class imbalance)

Recommendations:
  Revenue maximisation  → Decision Tree    (highest F1 + Recall, lowest FN)
  Cost-sensitive campaign → Naive Bayes    (highest Precision, lowest FP)
  Balanced robustness   → Hybrid AdaBoost  (best overall balance across all metrics)
""")